# Surrogate-Guided KAN (SG-KAN) Experiments

In [1]:
import torch
import gpytorch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import evaluation_metrics as em
import gaussian_process_models as gp
random_seed = 42

## STAGE 1: Create dataset and fit an Gaussian Process (GP) model

### Build the toy dataset
Define train/test sizes and create a 1D sinusoidal regression dataset with noise in training only.

In [2]:
# # TOY Dataset
# torch.manual_seed(random_seed)

# # ─── 1. Create dataset ───────────────────────────────────
# N_train = 100
# N_test  = 300

# # Input: uniform in [-3, 3]
# X_train = torch.linspace(-3, 3, N_train).unsqueeze(1)  # shape (100, 1)
# y_train = torch.sin(X_train.squeeze()) + 0.1 * torch.randn(N_train)

# X_test  = torch.linspace(-3, 3, N_test).unsqueeze(1)   # shape (300, 1)
# y_test  = torch.sin(X_test.squeeze())                   # noiseless ground truth

# print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
# print(f"X_test:  {X_test.shape},   y_test:  {y_test.shape}")

In [3]:
def create_barron_dataset(
        D: int = 1,
        N_train: int = 500,
        N_test: int = 2000,
    ):
    """Generate multi-dimensional Barron function dataset for testing approximation algorithms.
    
    Args:
        D: Input dimension (default: 1)
        N_train: Number of training samples (default: 500)
        N_test: Number of test samples (default: 2000)
    
    Returns:
        Tuple of (X_train, X_test, y_train, y_test) tensors
    """
    torch.manual_seed(random_seed)
    
    # Fixed constant vector a, shape (D,)
    a = torch.tensor([2*j/D - 1 for j in range(1, D+1)], dtype=torch.float32)
    
    # Input points uniformly sampled from [-1, 1]^D
    X_train = torch.rand(N_train, D) * 2 - 1
    X_test  = torch.rand(N_test,  D) * 2 - 1
    
    # Target function: f(x) = sqrt(3/2) * (||x - a|| - ||x + a||)
    # The Barron norm of f is equal to one for all input dimensions
    def barron(X):
        return torch.sqrt(torch.tensor(1.5)) * (
            torch.norm(X - a, dim=1) - torch.norm(X + a, dim=1)
        )
    
    y_train = barron(X_train)
    y_test  = barron(X_test)
    
    print(f"[DEBUG] Barron dataset created: D={D}, N_train={N_train}, N_test={N_test}, y_range=[{y_train.min():.3f}, {y_train.max():.3f}]")
    
    return X_train, X_test, y_train, y_test

# X_train, X_test, y_train, y_test = create_barron_dataset(D=2)
from datasets import load_dataset
data = load_dataset("TF1")
X_train, X_test, y_train, y_test = data["train_input"], data["test_input"], data["train_label"], data["test_label"]
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5000, 2), (10000, 2), (5000,), (10000,))

### Initialize the GP model
Create the GP model and likelihood that will be trained on the synthetic data.

In [5]:
X_train = torch.as_tensor(X_train, dtype=torch.float32)
y_train = torch.as_tensor(y_train, dtype=torch.float32)
y_test = torch.as_tensor(y_test, dtype=torch.float32)
X_test = torch.as_tensor(X_test, dtype=torch.float32)

In [6]:
# Initialize GP model
model, likelihood = gp.init_gp(X_train, y_train, num_inducing=500)

[DEBUG] Initialized SparseGPModel: X_train.shape=torch.Size([5000, 2]), num_inducing=500


### Train the GP
Optimize GP hyperparameters on the training set for a fixed number of iterations.

In [7]:
# Train GP model
model, likelihood = gp.train_gp(
    model,
    likelihood,
    X_train,
    y_train,
    num_iters=2000,
    lr=0.005
)

[DEBUG] SparseGPModel and likelihood unfrozen (train mode)
[DEBUG] Training SparseGPModel: 2000 iters, lr=0.005, loss=ELBO
  Iter 400/2000, Loss: 0.764467
  Iter 800/2000, Loss: 0.755712
  Iter 1200/2000, Loss: 0.752313
  Iter 1600/2000, Loss: 0.750285
  Iter 2000/2000, Loss: 0.749108
[DEBUG] Training complete.
  Lengthscale: tensor([[1.3063, 1.0209]])
  Outputscale: 0.9561
  Noise: 0.693247
  Mean const: -0.0207
[DEBUG] SparseGPModel and likelihood frozen (eval mode)


### Predict on the test grid
Compute the GP posterior mean and standard deviation on the test inputs.

In [8]:
# Predict using GP model
pred_test_mean, pred_test_std = gp.predict(model, likelihood, X_test)

[DEBUG] SparseGPModel and likelihood frozen (eval mode)


### Sanity-check shapes
Verify that train/test tensors and GP outputs align as expected.

In [9]:
X_test.shape, y_test.shape, pred_test_mean.shape, pred_test_std.shape

(torch.Size([10000, 2]),
 torch.Size([10000]),
 torch.Size([10000]),
 torch.Size([10000]))

### Evaluate GP on the test set
Compute MSE and relative $L_2$ error against noiseless targets.

In [10]:
# Evaluate GP model
mse_test = em.compute_mse(pred_test_mean, y_test)
rel_l2_test = em.compute_relative_l2(pred_test_mean, y_test)

# Print test results
print(f"  Test MSE: {mse_test.item():.6f}")
print(f"  Test Relative L2: {rel_l2_test.item():.6f}")

  Test MSE: 0.001320
  Test Relative L2: 0.035520


### Evaluate GP on the training set
Report training metrics to compare fit quality with the test results.

In [11]:
# Train set evaluations

pred_train_mean, pred_train_std = gp.predict(model, likelihood, X_train)

mse_train = em.compute_mse(pred_train_mean, y_train)
rel_l2_train = em.compute_relative_l2(pred_train_mean, y_train)

# Print results
print(f"  Train MSE: {mse_train.item():.6f}")
print(f"  Train Relative L2: {rel_l2_train.item():.6f}")

[DEBUG] SparseGPModel and likelihood frozen (eval mode)
  Train MSE: 0.001087
  Train Relative L2: 0.032968


## STAGE 2: Find informative regions of the input domain using GP model

### Sample candidate pairs
Draw $M$ random start/end pairs from the training data for SWIM-style scoring.

In [12]:
import gp_swim_like_pairs as gs

# Create M pairs from the training set like SWIM algorithm
x_a, x_b, y_a, y_b = gs.sample_candidate_pairs(X_train, y_train, M=500)

### Create interior points
Generate $T$ evenly spaced interior points between each pair to probe uncertainty.

In [13]:
# Create T interior point between x_a and x_b for posterior GP computation
T = 100
x_interior, x_interior_flat = gs.create_interior_points(x_a, x_b, T)

### Check interior shapes
Confirm the interior-point tensors have expected dimensions.

In [14]:
x_interior.shape, x_interior_flat.shape

(torch.Size([500, 100, 2]), torch.Size([50000, 2]))

### Score and select pairs
Compute GP-SWIM scores and sample the most informative pairs for the layer.

In [15]:
# Gradient scores and probabilities calculation
g_scores, g_probs = gs.compute_score_g(model, x_a, x_b, T=T)

# Select the most informative pairs
layer_width = 512
x_a_selected, x_b_selected, selected_idx = gs.select_pairs(x_a, x_b, g_probs, layer_width=layer_width)

[DEBUG] Scores: min=0.0082, max=0.6598, mean=0.3153
[DEBUG] Probs:  min=0.000052,  max=0.004186,  sum=1.000000
[DEBUG] Selected 512 pairs from 500 candidates
[DEBUG] Unique pairs selected: 300 / 512


## STAGE 3: Edge Function Construction

### Sample edge functions
Evaluate GP mean along each selected segment to build edge activations.

In [16]:
G = 1000 # Grid
import surrogate_guided_kan as sgkan

# Sample GP posterior mean functions over each selected pair segment
# Edge activation functions in the KAN-style network
x_segments, edge_functions = sgkan.sample_edge_functions(
    model,
    x_a_selected, x_b_selected,
    G_sample=G
)

print(x_segments.shape) 
print(edge_functions.shape) 

[DEBUG] x_segments shape:     torch.Size([512, 1000, 2])
[DEBUG] edge_functions shape: torch.Size([512, 1000])
[DEBUG] Value range: [-2.6273, 2.5176]
torch.Size([512, 1000, 2])
torch.Size([512, 1000])


### Build training features
Interpolate edge functions at training inputs to form $H_{train}$.

In [17]:
# Interpolate values
H_train = sgkan.interpolate_edge_functions(x_segments, edge_functions, X_train, x_a_selected, x_b_selected)

[DEBUG] H shape: torch.Size([5000, 512])
[DEBUG] H value range: [-2.6273, 2.5176]
[DEBUG] H rank (approx): 198


### Fit output layer
Solve the linear least-squares weights for the final layer.

In [18]:
W_out = sgkan.solve_output_layer(H_train, y_train)

[DEBUG] W_out shape: torch.Size([513, 1])


### Build test features
Interpolate edge functions at test inputs to form $H_{test}$.

In [19]:
H_test = sgkan.interpolate_edge_functions(x_segments, edge_functions, X_test, x_a_selected, x_b_selected)

[DEBUG] H shape: torch.Size([10000, 512])
[DEBUG] H value range: [-2.6273, 2.5176]
[DEBUG] H rank (approx): 139


### Evaluate SG-KAN
Predict on train/test features and report metrics.

In [20]:
y_pred_train, mse_train, rel_l2_train = sgkan.predict_and_evaluate(H_train, y_train, W_out, "Train")
y_pred_test,  mse_test,  rel_l2_test  = sgkan.predict_and_evaluate(H_test,  y_test,  W_out, "Test")
print("\n")

[Train] MSE: 0.004698 | Relative L2: 0.068541
[Test] MSE: 0.005699 | Relative L2: 0.073805




## Multi-layer SGKAN

### Configure a multi-layer SG-KAN
Define per-layer widths and sampling settings, then train and evaluate a stacked SG-KAN.

In [ ]:
layer_configs = [
    {"width": 50, "M": 1000, "G": 200, "T": 5},
    {"width": 20, "M": 500,  "G": 100, "T": 3},
]
activation = lambda x: x
layers, W_out = sgkan.build_sgkan(X_train, y_train, layer_configs)

y_pred_test  = sgkan.predict_sgkan(layers, W_out, X_test)
y_pred_train = sgkan.predict_sgkan(layers, W_out, X_train)

# Evaluate SGKAN
mse_test = em.compute_mse(y_pred_test, y_test)
rel_l2_test = em.compute_relative_l2(y_pred_test, y_test)

# Print test results
print(f"  Test MSE: {mse_test.item():.6f}")
print(f"  Test Relative L2: {rel_l2_test.item():.6f}")
    

### Single-layer baseline
Train and evaluate a one-layer SG-KAN for comparison.

In [ ]:
layer_configs = [
    {"width": 50, "M": 1000, "G": 200, "T": 5},
    # {"width": 20, "M": 500,  "G": 100, "T": 3},
]

layers, W_out = sgkan.build_sgkan(X_train, y_train, layer_configs)

y_pred_test  = sgkan.predict_sgkan(layers, W_out, X_test)
y_pred_train = sgkan.predict_sgkan(layers, W_out, X_train)

# Evaluate SGKAN
mse_test = em.compute_mse(y_pred_test, y_test)
rel_l2_test = em.compute_relative_l2(y_pred_test, y_test)

# Print test results
print(f"  Test MSE: {mse_test.item():.6f}")
print(f"  Test Relative L2: {rel_l2_test.item():.6f}")

### Scratch cell
Optional space for additional experiments or plots.